# Automated Onboarding/Offboarding System

## Libraries

In [1]:
from datetime import datetime
import json
import os
import pandas as pd

## Functions

### Detect Missing Data

In [17]:
def missing_data(data):
    missing = pd.isna(data)
    if isinstance(data, str):
        missing = not data.strip()
        
    return missing

### Logging Function

In [18]:
def log(level, message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open("logs.txt", "a") as f:
        f.write(f"[{timestamp}] [{level}] {message}\n")

## Load Permissions Info

In [19]:
roles_df = pd.read_excel("roles_permissions.xlsx")

role_permissions = {}

for _, row in roles_df.iterrows():
    role = row["role"]
    system = row["system"]
    access = row["access_level"]

    if role not in role_permissions:
        role_permissions[role] = []

    role_permissions[role].append({
        "system": system,
        "access": access
    })

print(json.dumps(role_permissions, indent=4))

{
    "Manager": [
        {
            "system": "Email",
            "access": "standard"
        },
        {
            "system": "Finance App",
            "access": "admin"
        }
    ],
    "Analyst": [
        {
            "system": "Email",
            "access": "standard"
        },
        {
            "system": "Finance App",
            "access": "read"
        }
    ]
}


## Load Truth JSONs

### Users

In [27]:
if os.path.exists("users.json"):
    with open("users.json", "r") as f:
        users = json.load(f)
else:
    users = {}

print(json.dumps(users, indent=4))

{
    "Paul Atreides": {
        "role": "Manager",
        "status": "active"
    },
    "Jessica Nerus": {
        "role": "HR",
        "status": "active"
    },
    "Duncan Idaho": {
        "role": "Analyst",
        "status": "active"
    }
}


### Access

In [28]:
if os.path.exists("access.json"):
    with open("access.json", "r") as f:
        access_data = json.load(f)
else:
    access_data = {}

print(json.dumps(access_data, indent=4))

{
    "Paul Atreides": [
        {
            "system": "Email",
            "access": "standard"
        },
        {
            "system": "Finance App",
            "access": "admin"
        }
    ],
    "Duncan Idaho": [
        {
            "system": "Email",
            "access": "standard"
        },
        {
            "system": "Finance App",
            "access": "read"
        }
    ]
}


## Get Requests

### Varilables for Execution Summary

In [23]:
completed = 0
failed = 0
skipped = 0

### Load Excel

In [24]:
df = pd.read_excel("requests.xlsx")

print(df)

    request_id             name    action     role approved     status
0            1              NaN   onboard  Manager      yes     failed
1            2                    onboard  Manager      yes     failed
2            3    Paul Atreides   onboard      NaN      yes     failed
3            4    Paul Atreides   onboard  Manager       no    pending
4            5    Paul Atreides   onboard  Manager      yes  completed
5            6    Paul Atreides   onboard  Manager      yes     failed
6            7   Gurney Halleck   onboard  Analyst      yes  completed
7            8    Jessica Nerus   onboard       HR      yes     failed
8            9     Duncan Idaho   onboard  Analyst      yes  completed
9           10  Wellington Yueh   unknown  Analyst      yes     failed
10          11    Alia Atreides  offboard  Analyst      yes    pending
11          12    Jessica Nerus  offboard       HR      yes    pending
12          13     Duncan Idaho  offboard  Analyst      yes    pending


### Pending Requests

In [25]:
pending_requests = df[
    (df["status"] == "pending") &
    (df["approved"] == "yes")
]

print(pending_requests)

    request_id           name    action     role approved   status
10          11  Alia Atreides  offboard  Analyst      yes  pending
11          12  Jessica Nerus  offboard       HR      yes  pending
12          13   Duncan Idaho  offboard  Analyst      yes  pending


### Skipped Requests

In [26]:
skipped_requests = df[
    (df["status"] == "pending") &
    (df["approved"] != "yes")
]

for _, row in skipped_requests.iterrows():
    skipped += 1
    log("WARNING", f"Skipped {row["name"]} — not approved")

print(skipped_requests)

   request_id           name   action     role approved   status
3           4  Paul Atreides  onboard  Manager       no  pending


## Loop to Update Employees

In [29]:
for index, row in pending_requests.iterrows():
    success = False
    try:
        # Get row data
        name = row["name"]
        action = row["action"]
        role = row["role"]

        # Onboarding
        if action == "onboard":
            # Add user
            if missing_data(name) or missing_data(role):
                log("ERROR", f"Missing data for request {row['request_id']}")
            elif name in users:
                log("INFO", f"Skipped {name} — already exists")
            else:
                users[name] = {
                    "role": role,
                    "status": "active"
                }
                log("INFO", f"Created user: {name}")
            
                # Grant access
                if role not in role_permissions:
                    log("ERROR", f"Role {role} not found for {name}")
                else:
                    permissions = role_permissions.get(role, [])
                    access_data[name] = permissions
                    log("INFO", f"Assigned permissions to {name}")

                    success = True

        # Offboarding
        elif action == "offboard":
            if name not in users:
                log("ERROR", f"Cannot offboard {name} — user does not exist")
            else:
                # Remove permissions
                if name in access_data:
                    del access_data[name]
                    log("INFO", f"Removed all access for {name}")
                else:
                    log("INFO", f"No access found for {name}")
    
                # Deactivate user
                users[name]["status"] = "inactive"
                log("INFO", f"Deactivated user: {name}")

                success = True

        else:
            log("ERROR", f"Unknown action {action} for {name}")

    except Exception as e:
        log("ERROR", f"Failed to process {row['name']}: {str(e)}")

    # Report success
    if success:
        completed += 1
        df.loc[index, "status"] = "completed"
    else:
        failed += 1
        df.loc[index, "status"] = "failed"

## Execution Summary

In [30]:
print("=== RUN SUMMARY ===")
print(f"Completed: {completed}")
print(f"Failed: {failed}")
print(f"Skipped: {skipped}")

=== RUN SUMMARY ===
Completed: 2
Failed: 1
Skipped: 1


## Update JSONs

### Users

In [31]:
with open("users.json", "w") as f:
    json.dump(users, f, indent=4)

print(json.dumps(users, indent=4))

{
    "Paul Atreides": {
        "role": "Manager",
        "status": "active"
    },
    "Jessica Nerus": {
        "role": "HR",
        "status": "inactive"
    },
    "Duncan Idaho": {
        "role": "Analyst",
        "status": "inactive"
    }
}


### Access

In [32]:
with open("access.json", "w") as f:
    json.dump(access_data, f, indent=4)

print(json.dumps(access_data, indent=4))

{
    "Paul Atreides": [
        {
            "system": "Email",
            "access": "standard"
        },
        {
            "system": "Finance App",
            "access": "admin"
        }
    ]
}


## Update Excel

In [33]:
df.to_excel("requests.xlsx", index=False)